In [ ]:
# Inspect Evidence Generation Pipeline

Read and analyze the generator, schema, and label logic to understand why "applicable_exception_types" is empty.

from pathlib import Path

root = Path(r'd:\Opencode')
paths = [
    'apx/evidence/generate_evidence.py',
    'apx/evidence/schemas.py',
    'apx/evidence/populate_eval_labels.py',
    'apx/exceptions/taxonomy.py',
    'apx/data/datasets/evidence/evidence_corpus.json',
    'apx/data/datasets/eval/eval_dataset.json',
]

for rel in paths:
    path = root / rel
    print(f'\n=== {rel} ===')
    if not path.exists():
        print('MISSING')
        continue
    text = path.read_text(encoding='utf-8')
    print(text[:2000])


# STEP 6C — REPAIR THE EVIDENCE-CORPUS GROUND-TRUTH METADATA

This notebook documents the controlled repair of the evidence-corpus applicability metadata.
It does not modify retrieval architecture; it repairs the semantic ground truth used to label evaluation data.

## Objective

Repair the evidence-corpus metadata generation so each evidence record has deterministic, semantically justified exception applicability derived from the evidence itself and the repository exception taxonomy.

## Requirements

- Keep retrieval implementation frozen.
- Preserve the repaired label semantics: temporal validity + vendor/scope compatibility + explicit exception applicability.
- Do not weaken the predicate or add randomness.
- Regenerate the corpus and eval dataset using the official generation path.
- Validate with focused tests, full pytest, and the dev benchmark.


## Section: Analyze Exception Taxonomy

Inspect the repository exception taxonomy and map each exception code to the evidence semantics it represents.

- `VENDOR_MISMATCH`: supplier/vendor identity mismatch.
- `PO_MISMATCH`: PO reference, procurement, or purchase-order validation mismatch.
- `AMOUNT_MISMATCH`: price, total, or amount tolerance mismatch.
- `GRN_MISMATCH`: receipt, quantity, or GRN reconciliation mismatch.
- `DUPLICATE_INVOICE`: duplicate invoice or repeated historical issue.
- `TAX_ERROR`: tax calculation or VAT/sales-tax discrepancy.
- `CURRENCY_MISMATCH`: invoice and contract currency mismatch.
- `LINE_ITEM_MISMATCH`: line quantity or item mismatch.
- `DISCOUNT_ERROR`: discount or early-payment rule violation.
- `CREDIT_ISSUE`: credit hold, credit limit, or payment-status issue.

The authoritative source remains the exception taxonomy in [apx/exceptions/taxonomy.py](apx/exceptions/taxonomy.py).